# 30.05 Масштабные законы двуслойной модели

> **Статус:** канонический синтетический анализ тождеств и асимптотик модели
> `30.01`. Экспериментальные данные и субъектные параметры не используются.

Цель — отделить свойства идеальной формулы от выводов о реальном эксперименте.
Notebook использует единственное ядро `two_layer_model.py`. Прежняя смешанная
версия сохранена в
`archive/legacy/30.91_Историческое_аналитическое_обоснование.ipynb`.

## 1. Безразмерная форма

При $eta=b/a$, $\eta=h/a$, $r=ho_2/ho_1$ и
$k=(r-1)/(r+1)$ формула `30.01` записывается как

$$
Z=rac{ho_1}{\pi a}\,\Psi(r,\eta,eta),
$$

где

$$
\Psi=\left(rac1{1-eta}-rac1{1+eta}ight)
+2\sum_{i=1}^{\infty}k^i\left[
rac1{\sqrt{(1-eta)^2+(2i\eta)^2}}-
rac1{\sqrt{(1+eta)^2+(2i\eta)^2}}
ight].
$$

Следовательно, одновременное масштабирование всех длин в $c$ раз изменяет
$Z$ в $1/c$ раз. Масштабирование обоих сопротивлений в $q$ раз изменяет $Z$ в
$q$ раз. Эластичности зависят от $(r,\eta,eta)$, а не от абсолютного размера.

In [ ]:
from pathlib import Path
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
notebook_root = next((p for p in candidates if (p / "two_layer_model.py").exists()), None)
if notebook_root is None:
    raise FileNotFoundError("two_layer_model.py not found; run from the repository or Colab Notebooks directory")
sys.path.insert(0, str(notebook_root))

from two_layer_model import evaluate, geometry_from_size, transfer_impedance

In [ ]:
rho1, rho2, beta = 6.0, 18.0, 0.5
for eta in (0.3, 0.6, 1.0, 2.0):
    values = []
    sensitivities = []
    for a in (0.050, 0.150):
        h, b = eta * a, beta * a
        result = evaluate(rho1, rho2, h, a, b)
        values.append(result.z * a / rho1)
        sensitivities.append(rho2 / result.z * result.d_rho2)
    assert np.allclose(values[0], values[1], rtol=1e-10, atol=1e-12)
    assert np.allclose(sensitivities[0], sensitivities[1], rtol=1e-10, atol=1e-12)
    print(eta, values[0], sensitivities[0])

## 2. Асимптотика большой относительной глубины

При $\eta\gg1$ разность двух ядер каждого изображения имеет ведущий порядок
$a b/(4i^3h^3)$. Поэтому поправка второго слоя и эластичность по $ho_2$
убывают как $\eta^{-3}$ при фиксированных $r$ и $eta$.

Это асимптотика идеальной плоской модели. Она не доказывает
неидентифицируемость конкретного добровольца: для этого нужны фактические
размеры, калибровка, ковариация и проверка расхождения модели в серии `31`.

In [ ]:
a = 0.050
etas = np.array([2.0, 4.0, 8.0, 16.0, 32.0])
s_rho2 = []
for eta in etas:
    result = evaluate(rho1, rho2, eta * a, a, beta * a)
    s_rho2.append(rho2 / result.z * result.d_rho2)
s_rho2 = np.asarray(s_rho2)
local_slopes = np.diff(np.log(s_rho2)) / np.diff(np.log(etas))
assert abs(local_slopes[-1] + 3.0) < 0.01
print("eta:", etas)
print("S_rho2:", s_rho2)
print("local log-log slopes:", local_slopes)

## 3. Когда $L_{min}\propto h$

Если критерий использует только безразмерную величину, например порог
$S_{ho_2}\ge S_*$ при фиксированных $r$ и $eta$, он задаёт одно и то же
условие на $\eta=2h/L$. Поэтому найденное $L_{min}/h$ постоянно.

Это не относится автоматически к абсолютному условию
$|\Delta Z|\ge c\sigma_Z$ с фиксированным $\sigma_Z$ в омах: при геометрически
подобном увеличении $h$ и $L$ сигнал уменьшается как $1/h$. Значения $S_*$,
$c$ и $\sigma_Z$ должны быть внешними требованиями, а не результатом модели.

In [ ]:
s_threshold = 0.25  # только синтетическая демонстрация масштабного тождества
lambda_grid = np.linspace(0.5, 40.0, 4000)  # lambda = L/h

def lmin_by_dimensionless_threshold(h):
    for ratio in lambda_grid:
        size = ratio * h
        a, b = geometry_from_size(size, beta)
        result = evaluate(rho1, rho2, h, a, b)
        sensitivity = rho2 / result.z * result.d_rho2
        if sensitivity >= s_threshold:
            return size
    raise RuntimeError("threshold is outside the scanned dimensionless range")

lmin_1 = lmin_by_dimensionless_threshold(0.010)
lmin_2 = lmin_by_dimensionless_threshold(0.030)
assert np.isclose(lmin_1 / 0.010, lmin_2 / 0.030, rtol=1e-12)

rho2_exhale, rho2_inhale = 15.0, 25.0
h1, size1, scale = 0.020, 0.140, 3.0
a1, b1 = geometry_from_size(size1, beta)
a2, b2 = geometry_from_size(scale * size1, beta)
delta_1 = transfer_impedance(rho1, rho2_inhale, h1, a1, b1) - transfer_impedance(rho1, rho2_exhale, h1, a1, b1)
delta_2 = transfer_impedance(rho1, rho2_inhale, scale * h1, a2, b2) - transfer_impedance(rho1, rho2_exhale, scale * h1, a2, b2)
assert np.isclose(delta_2, delta_1 / scale, rtol=1e-10)
print("Lmin/h:", lmin_1 / 0.010)
print("absolute signal ratio after x3 geometry:", delta_2 / delta_1)

## 4. Граница вывода

Внутри модели подтверждены только:

1. безразмерная редукция $(r,\eta,eta)$;
2. однородность по сопротивлениям и обратная однородность по длинам;
3. асимптотика $S_{ho_2}=O(\eta^{-3})$;
4. линейный закон $L_{min}\propto h$ только для безразмерного критерия.

CRLB, Fisher/SVD, распределения параметров, субъектные оценки, «ложное»
$\Deltaho_1$ и рекомендации расширить изготовленный диапазон не следуют из
этих тождеств сами по себе. Они должны проверяться отдельно в `31.02–31.04`,
`32` и `33` с реальными входами и явно заданной моделью ошибок.